## Hashing
* Hashing coverts a hashable object into an integer.
* An dictionary or set uses that integer to begin locating the object.
* Equal objects must have equal hashes.
* Unequal objects may have the same hash (collision).

In [ ]:
lst = ["john", "bill", "mary"]

'bill' in lst # complexity O(n)

# hashing idea today will allow us to do look up in O(1)

In [ ]:
1 == 1.0, hash(1) == hash(1.0), hash(1), hash(1.0)

(True, True, 1, 1)

In [ ]:
hash(-1), hash(-2) # collision

(-2, -2)

In [ ]:
# hashables:
#  int, float, str, bool, NoneType, range
hash(1), hash(1.2), hash('john'), hash(True), hash(None), hash(range(5))

(1,
 461168601842738689,
 -6446521346671130196,
 1,
 4238894112,
 5795932985296280846)

In [ ]:
# possibly hashable: tuple
# tuple is hashable only when its elements are all hashable
hash((1, 2)) # yes
#hash((1, [2])) # no

-3550055125485641917

In [ ]:
# unhashable: list, dict, set
#hash([]) # an empty list
#hahs({}) # an empty dict
#hash(set()) # an empty set

### Hashing integers

In [ ]:
# For most ordinary small integers, the hash is the integer itself.
hash(5) == 5
hash(-10) == -10

# Exception: Python reserves -1 internally as an error indicator,
# so the hash value -1 is replaced with -2.
hash(-1) == -2

In [ ]:
# FYI, not required
import sys

P = sys.hash_info.modulus
P, P == 2**61 - 1

# Conceptually, for a nonnegative integer n:
#
#     hash(n) = n % P
#
# Therefore, if 0 <= n < P:
#
#     hash(n) == n
#
# Numbers that differ by P normally have the same hash:
#
#     hash(n + P) == hash(n)

x = 123456

print(hash(x) == x)
print(hash(P - 1) == P - 1)
print(hash(P + 5) == hash(5))

In [ ]:
def conceptual_integer_hash(n):
    if n >= 0:
        result = n % P
    else:
        result = -((-n) % P)

    if result == -1:
        result = -2

    return result

# Note: The -1 case is special because Python reserves -1
# internally as an error indicator.

### Hashing floats

In [ ]:
x = 3.4
x.as_integer_ratio()

(7656119366529843, 2251799813685248)

In [ ]:
1 == 1.0, hash(1) == hash(1.0)

(True, True)

In [ ]:
x = 1.0
x.as_integer_ratio()

(1, 1)

### Hashing strings
* Python’s string-hashing algorithm processes the characters—their Unicode code-point information—in order, together with a secret key.
* A secret key is a randomly generated value that Python chooses when a Python process starts.
* Therefore, separate notebook sessions may produce different hash values for the same string.
* Within one running session, the same string always produces the same hash value.

In [ ]:
s = 'hello'

# character -> code point (it is a number)
ord('h')

104

In [ ]:
[ord(c) for c in s]

[104, 101, 108, 108, 111]

### Hashing tuples

In [ ]:
hash((5, 'hello', 3.4))

9109462883715067864

## Hash table
* A hash table is an array of fixed-size slots.
    * Each occupied slot stores entry information, including references to the key and, for a dictionary, the value.
    * The referenced objects can have different sizes.
* Hash tables use hashing and collision resolution to provide expected O(1) lookup, insertion, and deletion.
    * linear probing (a type of open addressing)
* open addressing vs chaining
* When hash table is full, python allocates a larger array
* We assume that the table does not store duplicate keys.
    

### Insertion

In [ ]:
# example of open addressing, in particular, linear probing

# hash table:
capacity = 7

# inserting keys

# hashing name
# hash(name) % capacity ==> first position to lookup

pos = hash("john") % capacity
print(pos)  # For this example, assume the result is 4.
# position 4 is free, so put "john" there

pos = hash('david') % capacity # assume pos is 4
# 4 is occupied, try pos + 1 == 5
# 5 is free, put "david" there

hash('mary') % capacity # assume the result is 6

hash("bill") % capacity # assume 1

hash('jack') % capacity # assume 3

pos = hash('alice') % capacity # 6
# try pos == 6, occupied
# try (pos + 1) % capacity ==> 0
# 0 is free, put "alice" there

# Index: 0         1    2     3      4        5       6
# Value:"alice" "bill"      'jack' "john"  "david"   "mary"

3


In [ ]:
# resolve collisions using chaining

# example of chaining
# Index: 0      1    2     3      4        5       6
# Value:      ["bill"]   ['jack'] ["john"]

# when hashing "david" -> pos 4 as well
# hash table pos 4 will have ['john', 'david']

In [ ]:
# capacity = 7
# full
# capactity = 15

# suppose hash(x) = 21 where x is a name
# before expansion,
#    pos = 21 % 7 = 0
# after expansion
#    pos = 21 % 15 = 6
21 % 15

# suppose hash table has n elements,
# what is the cost of rehashing (into ...)?
# O(n)

6

**Load factor**

The load factor measures how full a hash table is:

$$
\text{load factor}
=
\frac{\text{number of active entries}}
{\text{table capacity}}
$$

As the load factor increases, collisions become more frequent and
probing sequences tend to become longer.

Hash tables expand before becoming excessively full. After expansion,
existing entries must be repositioned because their indices depend on
the table capacity:

```python
index = hash(key) % capacity

**Complexity of insertion**

- Expected case: \(O(1)\)
  - Hash values are normally well distributed, so insertion requires
    probing only a small number of slots.
- Worst case: \(O(n)\)
  - An insertion may trigger table expansion.
  - Severe collisions may require probing many slots.

**Note:** When expansion costs are distributed across a long sequence
of insertions, insertion is more precisely described as having
**expected amortized \(O(1)\)** complexity.

### Lookup

In [ ]:
# Lookup example

# Index:    0      1      2     3      4        5       6
# Value:  "alice" "bill"      'jack' "john"  "david"   "mary"
#

hash('jack') % capacity # found "jack" in position 3

hash('david') % capacity # go to position 4, 'david' is not there
# try position (4 + 1) % capacity = 5, found 'david' there

hash('jennifer') % capacity # go to position 5, found "david"
# go to position 6, found "mary"
# go to position 0, found "alice"
# go to position 1, found "bill"
# go to position 2, found unused slot, stop!


**Complexity of lookup**
* Expected case: O(1)
* Worst case: O(n), when collisions require probing many slots
    
* The worst case can occur for either:
    * A present key located near the end of a long probing sequence.
    * An absent key that requires examining many slots before its absence can be established.


### Deletion

In [ ]:
# Index:    0      1      2     3      4        5       6
# Value:  "alice" "bill"      'jack' DELETED "david"  DELETED
#

# deleting "john"

hash('john') % capacity # lookup in pos 4, found "john"
#  remove reference of "john", put "DELETED" mark there

hash('mary') % capacity # look up in pos 6, found "mary"
#  remove reference of "mary", put "DELETED" mark there

hash('jennifer') % capacity # look up in pos 5, found 'david'
#  continue to pos 6, found DELETED
#  continue to pos 0, found 'alice'
#  continue to pos 1, found 'bill'
#  continue to pos 2, found empty slot, stop

# suppose pos 2 is also occupied, it will continue to pos 3
# since pos 3 is also occupied, continue to pos 4,
#    found "DELETED" mark, stop

# we looked up pos: 5, 6, 0, 1, 2, 3, 4

5

**Complexity of deletion**
* Expected case: O(1)
* Worst case: O(n), when collisions require probing many slots

### Algorithms & complexity summary:
* Search

    1. Use the key’s hash to determine where to begin.
    2. Examine table positions according to the probing sequence.
    3. Stop when the key is found or when the table establishes that the key is absent.

* Two-pass insertion

    1. Search to determine whether the key already exists.
    2. If it does not, search again for an available position and insert it.

* Deletion

    1. Search for the key.
    2. If found, mark its position as deleted.

A deleted position must not end a search, because another key may have been placed later in the same probing sequence.

Complexity summary:
| Hash-table operation | Expected case | Worst case | Reason for worst case |
|---|---:|---:|---|
| Lookup | \(O(1)\) | \(O(n)\) | Collisions require probing many slots |
| Insertion | \(O(1)\) | \(O(n)\) | Long probing sequence or table expansion |
| Deletion | \(O(1)\) | \(O(n)\) | Collisions require probing many slots to find the key |

Assumptions:
- Hash values are reasonably well distributed.
- The load factor is controlled.
- Computing a hash and comparing two keys each cost \(O(1)\).

In [ ]:
# search(table, key) => index or None
# insert(table, key):
#    search(table, key): // O(1), O(n)
#    if found: nothing to do
#    if not found:
#          put key in the first deleted/empty slot // O(1), O(n)


# expected: O(1)
# worst: O(n) // O(n) + O(n)

## Dictionaries

- Python implements dictionaries using hash tables.
- Each entry associates a key with a value.
- Python hashes the key to begin locating the entry.
- Only keys must be hashable.
- Values may be any Python objects.


In [ ]:
scores = {"john": 95, "bill": 89, "alice": 100}

In [ ]:
# Index:    0        1      2     3      4        5       6
# Key:   "alice"   "bill"             "john"
# Value:   100      89                  95

# Every table position is a fixed-size slot.
# Each slot is either EMPTY, DELETED, or contains references to a key and a value.

In [ ]:
# Dictionary lookup
scores["jack"]

# 1. Compute the initial position:
#       pos = hash("jack") % capacity
#
# 2. Starting from pos, follow the probing sequence:
#       * If "jack" is found, return its value.
#       * If an EMPTY slot is reached, "jack" is absent;
#         raise a KeyError.
#       * Continue past a DELETED slot.
#       * If every slot is checked, "jack" is absent;
#         raise a KeyError.
#
# Complexity:
#       Expected: O(1)
#       Worst case: O(n)

In [ ]:
# Simplified two-pass insertion/update model used in HW4

scores["jack"] = 90

# Pass 1: Search for "jack".
#
# If "jack" is found:
#   Replace its value with 90.
#
# If "jack" is not found:
#   Pass 2: Probe again to find an available slot.
#   Store references to "jack" and 90 in that slot.

# Complexity of dictionary insertion/update:
#
# Expected case: O(1)
#   * Hash values are normally well distributed.
#   * Only a small number of slots need to be probed.
#
# Worst case: O(n)
#   * Severe collisions may require probing many slots.
#   * Inserting a new key may trigger table expansion.
#
# More precisely, insertion has expected amortized O(1) cost
# across a sequence of insertions.

In [ ]:
## Dictionary deletion
del scores["jack"]

# One-pass deletion:
# 1. Search for 'jack'.
# 2. If found, mark its slot as DELETED
#    If not found, raise KeyError
#

# Complexity:
#       Expected: O(1)
#       Worst case: O(n)

In [ ]:
len(scores)

3

| Operation | Example | Expected | Worst case |
|---|---|---:|---:|
| Lookup | `scores[key]` | \(O(1)\) | \(O(n)\) |
| Safe lookup | `scores.get(key)` | \(O(1)\) | \(O(n)\) |
| Key membership | `key in scores` | \(O(1)\) | \(O(n)\) |
| Insert new key | `scores[key] = value` | \(O(1)\) | \(O(n)\) |
| Update existing key | `scores[key] = value` | \(O(1)\) | \(O(n)\) |
| Delete key | `del scores[key]` | \(O(1)\) | \(O(n)\) |
| Number of entries | `len(scores)` | \(O(1)\) | \(O(1)\) |
| Create keys view | `scores.keys()` | \(O(1)\) | \(O(1)\) |
| Create values view | `scores.values()` | \(O(1)\) | \(O(1)\) |
| Create items view | `scores.items()` | \(O(1)\) | \(O(1)\) |
| Key membership in keys view | `key in scores.keys()` | \(O(1)\) | \(O(n)\) |
| Value membership in values view | `value in scores.values()` | \(O(n)\) | \(O(n)\) |
| Item membership in items view | `(key, value) in scores.items()` | \(O(1)\) | \(O(n)\) |
| Iterate over all keys | `for key in scores` | \(O(n)\) | \(O(n)\) |
| Iterate over all values | `for value in scores.values()` | \(O(n)\) | \(O(n)\) |
| Iterate over all items | `for key, value in scores.items()` | \(O(n)\) | \(O(n)\) |
| Remove all entries | `scores.clear()` | \(O(n)\) | \(O(n)\) |

In [ ]:
len(scores)

# Conceptually, a dictionary stores:
#   * its number of active entries
#   * information about its table capacity
#   * a reference to its internal hash table
#
# Each table position is EMPTY, DELETED, or OCCUPIED.
# An occupied entry contains a key reference, a value reference,
# and typically the key's stored hash value.
#
# Because the active-entry count is stored, len(d) is O(1).

3

In [ ]:
keys = scores.keys()
values = scores.values()
items = scores.items()
type(keys)

dict_keys

In [ ]:
'''
scores ────────────────────────────────┐
                                      ▼
                            dictionary object
                         ┌─────────────────────┐
keys ──→ dict_keys view ─┤                     │
                         │  length = 3         │
values → dict_values view┤                     │
                         │  internal table ──┐ │
items ──→ dict_items view┤                  │ │
                         └───────────────────│─┘
                                             ▼
                                  hash-table entries
                         ┌──────────────────────────┐
                         │ "alice" reference → 89   │
                         │ "bill"  reference → 89   │
                         │ "john"  reference → 95   │
                         └──────────────────────────┘
'''

In [ ]:
scores

{'john': 95, 'bill': 89, 'alice': 100}

In [ ]:
scores['john'] = -1

In [ ]:
scores

{'john': -1, 'bill': 89, 'alice': 100}

In [ ]:
list(values)

[-1, 89, 100]

In [ ]:
'john' in keys # O(1), why?

True

In [ ]:
-1 in values # O(n)

True

In [ ]:
('john', -1) in items # O(1), why?

True

## Sets

In [ ]:
students = {"alice", "bill", "john"}

In [ ]:
tas = {"bill", "mary"}

In [ ]:
students | tas

{'alice', 'bill', 'john', 'mary'}

In [ ]:
students - tas

{'alice', 'john'}

In [ ]:
'''
students
    │
    ▼
set object
    │
    ├── length = 3
    └── internal hash table
            ├── reference → "alice"
            ├── reference → "bill"
            └── reference → "john"
'''

'\nstudents\n    │\n    ▼\nset object\n    │\n    ├── length = 3\n    └── internal hash table\n            ├── reference → "alice"\n            ├── reference → "bill"\n            └── reference → "john"\n'

| Operation | Example | Expected | Worst case |
|---|---|---:|---:|
| Membership | `x in s` | \(O(1)\) | \(O(n)\) |
| Add | `s.add(x)` | \(O(1)\) | \(O(n)\) |
| Remove | `s.remove(x)` | \(O(1)\) | \(O(n)\) |
| Discard | `s.discard(x)` | \(O(1)\) | \(O(n)\) |
| Length | `len(s)` | \(O(1)\) | \(O(1)\) |
| Iterate | `for x in s` | \(O(n)\) | \(O(n)\) |
| Copy | `s.copy()` | \(O(n)\) | \(O(n)\) |
| Union | `a \| b` | \(O(n+m)\) | — |
| Intersection | `a & b` | \(O(min(n,m))\) | — |
| Difference | `a - b` |  \(O(n)\) | — |
| Symmetric difference | `a ^ b` | \(O(n+m)\) | — |
| Subset test | `a <= b` |  \(O(n)\) | — |

where n = |a| and m = |b|.


In [ ]:
students, tas, students ^ tas # O(n + m) where n = |students|, m = |tas|

({'alice', 'bill', 'john'}, {'bill'}, {'alice', 'john'})

In [ ]:
tas < students # O(n) where n = |tas|

True

In [3]:
# Expected complexity: O(min(n, m))
# where n = len(a) and m = len(b)

def intersection(a, b):
    result = set()

    # Make a refer to the smaller set.
    if len(a) > len(b):
        a, b = b, a

    for x in a:          # min(n, m) iterations
        if x in b:       # expected O(1)
            result.add(x)  # expected O(1)

    return result

In [4]:
a = {1, 2, 3, 4}
b = {2, 3, 4, 5}
intersection(a, b)

{2, 3, 4}

In [6]:
# Set union: elements in a or b
# Expected complexity: O(n + m)

def union(a, b):
    result = a.copy()   # O(n)

    for x in b:         # m iterations
        result.add(x)   # expected O(1)

    return result

{1, 2, 3, 4, 5}

In [7]:
# Set difference: elements in a but not in b
# Expected complexity: O(n), where n = len(a)

def difference(a, b):
    result = set()

    for x in a:             # n iterations
        if x not in b:      # expected O(1)
            result.add(x)   # expected O(1)

    return result

{1}

## Heaps and priority queues

* Dictionaries and sets provide fast lookup, but what if we repeatedly need the smallest or highest-priority item?

* Possible list implementations have a tradeoff:
| List strategy |   Insert | Remove minimum |
| ------------- | -------: | -------------: |
| Unsorted list | \(O(1)\) |       \(O(n)\) |
| Sorted list   | \(O(n)\) |       \(O(1)\) |

* A priority queue uses a heap to provide a balance:
    * insertion and removal of the minimum both take $O(\log n)$
    * while examining the minimum takes \$O(1)$.

### Heap
* A heap is conceptually a complete binary tree.
* A complete binary tree has:
    * every level full except possibly the last;
    * nodes on the last level filled from left to right.
* A heap is commonly stored compactly in a list.
* A heap can contain any mutually comparable elements, not only integers.

### Min-heap
* Every parent is less than or equal to its children.
    * parent ≤ children
* Therefore, the root is the smallest element.



In [ ]:
#        1(0)
#      /    \
#     3(1)     2(2)
#    /   \     /
#   7(3) 5(4)  6(5)  <-- numbers in parenthesis are indexes

In [ ]:
# list representation
heap = [1, 3, 2, 7, 5, 6]

### Max-heap
* Every parent is greater than or equal to its children.
    * parent ≥ children
* Therefore, the root is the largest element.

In [ ]:
#        9
#      /   \
#     7     8
#    / \   /
#   3   5  6

### List-index relationships


In [ ]:
# for a node at index i, e.g.,
i = 1

# there are i nodes before node i,
# their children + root (i.e., 2*i + 1 nodes) must be before first child of i
left_child = 2 * i + 1
right_child = 2 * i + 2
parent = (i - 1) // 2

left_child, right_child, parent

(3, 4, 0)

In [ ]:
# assume n >= 2
# the last internal node must be the parent of the last node (index = n-1)

heap = [1, 3, 2, 7, 5, 6]
n = len(heap)
# since parent = (i - 1) // 2

# the index of last internal node must be:
(n - 1 - 1) // 2 == (n-2) // 2 == n//2 - 1

# note:
# (n-2)//2 = floor((n-2)/2) = floor(n/2 - 1) = floor(n/2) - 1
# (1 is an integer and can be moved outside floor)

# In other words,
# there are n//2 internal nodes, and n - n//2 = ceil(n/2) leaf nodes

True

### Heap operations and heapq library

In [ ]:
import heapq

# heap is stored in a list (which needs to be heapified first)
# heapq implements a min-heap by default
heap = [4, 1, 7, 3, 8, 5]

#### Heapify: O(n)

In [ ]:
heapq.heapify(heap)
heap

In [ ]:
# before heapify
#
#           4
#         /   \
#        1     7
#       / \   /
#      3   8 5
#
# list representation:
# [4, 1, 7, 3, 8, 5]


# Heapification:
#
# Start from the last internal node and continue backward to the root.
# For each internal node, fix the subtree rooted at that node by
# sifting its value downward.


# 1. Start with 7, the last internal node.
#
#           4
#         /   \
#        1     7
#       / \   /
#      3   8 5
#
# 7 > 5, so swap 7 with its smaller child, 5.
#
#           4
#         /   \
#        1     5
#       / \   /
#      3   8 7
#
# list:
# [4, 1, 5, 3, 8, 7]


# 2. Move backward to 1.
#
#           4
#         /   \
#        1     5
#       / \   /
#      3   8 7
#
# 1 <= 3 and 1 <= 8, so this subtree already satisfies
# the min-heap property. No change is needed.


# 3. Move backward to the root, 4.
#
#           4
#         /   \
#        1     5
#       / \   /
#      3   8 7
#
# Compare 4 with its children, 1 and 5.
# The smaller child is 1, so swap 4 and 1.
#
#           1
#         /   \
#        4     5
#       / \   /
#      3   8 7
#
# list:
# [1, 4, 5, 3, 8, 7]


# Continue sifting 4 downward.
#
# Compare 4 with its children, 3 and 8.
# The smaller child is 3, so swap 4 and 3.
#
#           1
#         /   \
#        3     5  # n/4
#       / \   /  \
#      4   8 7   6  # n/2
#
# final heap:
# [1, 3, 5, 4, 8, 7]

In [ ]:
# The cost of heapify.
#
# There are about n/2 leaves, which do not need to be fixed.
# The nodes immediately above the leaves can move down at most 1 level.
# The next group can move down at most 2 levels, and so on.
#
# Approximate number of nodes and maximum fixing cost:
#
# number of nodes       cost per node
# n/2                    0
# n/4                    1
# n/8                    2
# n/16                   3
# ...
# n / 2**k               k - 1
#
# Therefore, the total work is at most:
#
# T(n) = (n/2)*0 + (n/4)*1 + (n/8)*2 + (n/16)*3 + ...
#
#      = n * (1/4 + 2/8 + 3/16 + ...)
#
# The infinite series
#
# 1/4 + 2/8 + 3/16 + ...
#
# converges to 1. Therefore:
#
# T(n) <= n
#
# Thus:
#
# heapq.heapify(values) has O(n) time complexity.

#### Insert an element (heappush): O(log n)

In [ ]:
heapq.heappush(heap, 2)
heap
# Conceptually, first add 2 to the end of the list.
#
#           1
#         /   \
#        3     5
#       / \   / \
#      4   8 7   2  <--- new value
#
# [1, 3, 5, 4, 8, 7, 2]


# Sift 2 upward.
#
# Compare 2 with its parent, 5.
# Since 2 < 5, swap them.
#
#           1
#         /   \
#        3     2
#       / \   / \
#      4   8 7   5
#
# [1, 3, 2, 4, 8, 7, 5]


# Now compare 2 with its new parent, 1.
#
# 2 >= 1
#
# The heap property is satisfied, so stop.

[1, 3, 2, 4, 8, 7, 5]

* The number of levels in a complete binary tree is $O(\log n)$.
* In the worst case, the new value moves from a leaf to the root:

#### Remove the root (heappop): O(log n)

In [ ]:
heap

[1, 3, 2, 4, 8, 7, 5]

In [ ]:
x = heapq.heappop(heap)
x

# Before removal:
#
#           1  <--- minimum
#         /   \
#        3     2
#       / \   / \
#      4   8 7   5
#
# [1, 3, 2, 4, 8, 7, 5]


# Remove 1 and move the last value, 5, to the root.
#
#           5  <--- sift this value downward
#         /   \
#        3     2
#       / \   /
#      4   8 7
#
# [5, 3, 2, 4, 8, 7]


# Compare 5 with its children, 3 and 2.
# Swap it with the smaller child, 2.
#
#           2
#         /   \
#        3     5
#       / \   /
#      4   8 7
#
# [2, 3, 5, 4, 8, 7]


# Continue sifting 5 downward.
#
# Its only child is 7.
#
# 5 <= 7
#
# The heap property is satisfied, so stop.

# Because the replacement value can move through at most the
# height of the tree, the complexity of heappop is O(log n).

1

In [ ]:
heap

[2, 3, 5, 4, 8, 7]

#### Other operations and heap sort

In [ ]:
# inspect minimum: O(1)
heap[0]

# search for arbitrary values: O(n)
5 in heap

True

In [ ]:
# repeatedly heappop produces sorted order
heap = [4, 1, 7, 3, 8, 5]
heapq.heapify(heap)

while heap:
    print(heapq.heappop(heap))

# heapsort: O(n log n)

1
3
4
5
7
8


#### heap and priority queue

A priority queue supports three main operations:
* insert an item
* inspect the highest-priority item
* remove the highest-priority item

With a min-heap, the smallest value has the highest priority:

| Priority-queue operation | Heap operation |    Complexity |
| ------------------------ | -------------- | ------------: |
| Insert                   | `heappush`     | \(O(log n)\) |
| Inspect next             | `heap[0]`      |      \(O(1)\) |
| Remove next              | `heappop`      | \(O(log n)\) |


### Complexity summary

| Operation                     |    Complexity |
| ----------------------------- | ------------: |
| Examine minimum/maximum root  |      O(1) |
| Search for an arbitrary value |      O(n) |
| Insert                        | O($\log n)$ |
| Remove the root               | O($\log n$) |
| Build a heap with `heapify`   |      O(n) |


## HW4

In [ ]:
s = ' hello world '
s.strip() # remove leading and trailing whitespaces

'hello world'

In [ ]:
# s.split(" ") uses the literal space character as the separator.
# Leading and trailing spaces therefore produce empty strings.
# s.split() with no argument treats any consecutive whitespace
# as one separator and ignores leading and trailing whitespace.
s = ' hello  world '
s.split(" "), s.split()

(['', 'hello', '', 'world', ''], ['hello', 'world'])

In [ ]:
s = 'john, 25'
s.split(', ')

['john', '25']